In [ ]:
# 1. KUTUPHANELER VE VERI SETI OKUMA

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

dosya_yolu = 'Incident_Management_CSV.csv'
df = pd.read_csv(dosya_yolu, sep=';')

print('Veri seti yuklendi')
print('-' * 60)
print(f'Veri seti boyutu: {df.shape[0]} satir, {df.shape[1]} sutun')
display(df.head())


In [ ]:
# 2. SENA ON ISLEME AKISIYLA UYUMLU FEATURE ENGINEERING

df['Resolver'] = df['Resolver'].fillna('Unassigned')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Timestamp'])
df = df.sort_values(by=['Case ID', 'Timestamp'])

# her dosyanin ilk ve son hareketi arasindaki sure hedef degisken olarak kullanilir.
sure_df = df.groupby('Case ID')['Timestamp'].agg(['min', 'max'])
sure_df['Resolution_Time_Hours'] = (sure_df['max'] - sure_df['min']).dt.total_seconds() / 3600.0

# projenin siniflandirma akisi ile tutarli kalmak icin IQR ile ekstrem sureleri temizliyoruz.
Q1 = sure_df['Resolution_Time_Hours'].quantile(0.25)
Q3 = sure_df['Resolution_Time_Hours'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

sure_df_temiz = sure_df[
    (sure_df['Resolution_Time_Hours'] >= alt_sinir) &
    (sure_df['Resolution_Time_Hours'] <= ust_sinir)
]

df_temiz = df[df['Case ID'].isin(sure_df_temiz.index)]

step_counts = df_temiz.groupby('Case ID').size().rename('Step_count')
reassignment_counts = df_temiz.groupby('Case ID')['Resolver'].nunique().rename('Reassignment_count')
escalation_counts = df_temiz[df_temiz['Event'].astype(str).str.contains('escalat', case=False, na=False)].groupby('Case ID').size().rename('Escalation_count')
bounce_counts = df_temiz[df_temiz['Event'].astype(str).str.contains('reject|reopen', case=False, na=False)].groupby('Case ID').size().rename('Bounce_count')

regression_df = df_temiz.groupby('Case ID').agg({
    'Variant': 'first',
    'Priority': 'first',
    'Issue Type': 'first',
    'Report Channel': 'first',
    'Timestamp': 'min'
})

regression_df = regression_df.join([step_counts, reassignment_counts])
regression_df = regression_df.join(escalation_counts).fillna({'Escalation_count': 0})
regression_df = regression_df.join(bounce_counts).fillna({'Bounce_count': 0})
regression_df['Has_Bounce'] = (regression_df['Bounce_count'] > 0).astype(int)
regression_df['Workload_Index'] = regression_df['Step_count'] * regression_df['Reassignment_count']
regression_df['Open_Hour'] = regression_df['Timestamp'].dt.hour
regression_df['Open_Day'] = regression_df['Timestamp'].dt.dayofweek
regression_df['Is_Weekend'] = regression_df['Open_Day'].apply(lambda x: 1 if x >= 5 else 0)
regression_df = regression_df.join(sure_df_temiz[['Resolution_Time_Hours']])

regression_df = regression_df.drop(['Timestamp', 'Open_Day', 'Bounce_count'], axis=1)
regression_df = regression_df.dropna()

print('Regresyon modeli icin veri hazirlandi')
print('-' * 60)
print(f'Model verisi boyutu: {regression_df.shape[0]} dosya, {regression_df.shape[1]} sutun')
print('Hedef degisken ozeti:')
display(regression_df['Resolution_Time_Hours'].describe())
display(regression_df.head())


In [ ]:
# 3. MODEL GIRISLERI VE RANDOM FOREST REGRESYON PIPELINE'I

categorical_features = ['Variant', 'Priority', 'Issue Type', 'Report Channel']
numeric_features = [
    'Step_count', 'Reassignment_count', 'Escalation_count',
    'Has_Bounce', 'Workload_Index', 'Open_Hour', 'Is_Weekend'
]
model_features = categorical_features + numeric_features

X = regression_df[model_features]
y = regression_df['Resolution_Time_Hours']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

try:
    one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

try:
    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', 'passthrough', numeric_features),
            ('categorical', one_hot_encoder, categorical_features)
        ],
        remainder='drop',
        verbose_feature_names_out=False
    )
except TypeError:
    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', 'passthrough', numeric_features),
            ('categorical', one_hot_encoder, categorical_features)
        ],
        remainder='drop'
    )

duration_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

print('Egitim seti boyutu:', X_train.shape)
print('Test seti boyutu:', X_test.shape)


In [ ]:
# 4. MODELI EGITME VE REGRESYON METRIKLERI

duration_model.fit(X_train, y_train)
y_pred = duration_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('--- COZUM SURESI TAHMIN MODELI PERFORMANSI ---')
print(f'MAE  : {mae:.2f} saat')
print(f'RMSE : {rmse:.2f} saat')
print(f'R2   : {r2:.4f}')

sonuc_df = pd.DataFrame({
    'Gercek_Sure_Saat': y_test.values,
    'Tahmin_Sure_Saat': y_pred,
    'Mutlak_Hata_Saat': np.abs(y_test.values - y_pred)
})

print('\nOrnek tahminler:')
display(sonuc_df.head(10))


In [ ]:
# 5. GORSEL DEGERLENDIRME

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.45)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel('Gercek Cozum Suresi (Saat)')
plt.ylabel('Tahmin Edilen Cozum Suresi (Saat)')
plt.title('Gercek vs Tahmin Edilen Cozum Suresi')
plt.show()

residuals = y_test.values - y_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=30, kde=True)
plt.xlabel('Tahmin Hatasi (Saat)')
plt.title('Regresyon Modeli Hata Dagilimi')
plt.show()

feature_names = duration_model.named_steps['preprocessor'].get_feature_names_out()
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': duration_model.named_steps['regressor'].feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance.head(15), x='Importance', y='Feature')
plt.title('Cozum Suresi Tahmininde En Etkili Ozellikler')
plt.show()

display(feature_importance.head(15))


In [ ]:
# 6. STREAMLIT ARAYUZU ICIN MODEL VE VERI CIKTILARINI KAYDETME

joblib.dump(duration_model, 'rf_duration_regression_model.pkl')

duration_cases_df = regression_df[model_features + ['Resolution_Time_Hours']].copy()
duration_cases_df.insert(0, 'Case ID', regression_df.index)
duration_cases_df['Predicted_Duration_Hours'] = duration_model.predict(duration_cases_df[model_features])
duration_cases_df['Absolute_Error_Hours'] = (
    duration_cases_df['Resolution_Time_Hours'] - duration_cases_df['Predicted_Duration_Hours']
).abs()

regression_df.to_csv('SLA_Sure_Tahmin_Veri.csv', index=True)
duration_cases_df.to_csv('SLA_Sure_Tahmin_Arayuz_Dosyalari.csv', index=False)

duration_model_config = {
    'model_file': 'rf_duration_regression_model.pkl',
    'training_file': 'SLA_Sure_Tahmin_Veri.csv',
    'case_file': 'SLA_Sure_Tahmin_Arayuz_Dosyalari.csv',
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'model_features': model_features,
    'target': 'Resolution_Time_Hours',
    'prediction_column': 'Predicted_Duration_Hours',
    'metrics': {
        'MAE': float(mae),
        'RMSE': float(rmse),
        'R2': float(r2)
    }
}

with open('duration_model_config.json', 'w', encoding='utf-8') as f:
    json.dump(duration_model_config, f, ensure_ascii=False, indent=2)

print('Model kaydedildi: rf_duration_regression_model.pkl')
print('Regresyon egitim verisi kaydedildi: SLA_Sure_Tahmin_Veri.csv')
print('Arayuz sure tahmin dosyasi kaydedildi: SLA_Sure_Tahmin_Arayuz_Dosyalari.csv')
print('Model konfigurasyonu kaydedildi: duration_model_config.json')
display(duration_cases_df.head())

#indirme islemi:
# from google.colab import files
# files.download('rf_duration_regression_model.pkl')
# files.download('SLA_Sure_Tahmin_Veri.csv')
# files.download('SLA_Sure_Tahmin_Arayuz_Dosyalari.csv')
# files.download('duration_model_config.json')
